# Adaptive Safety Margins (ASM): Curriculum Approach to Recovery RL

**Research Project Implementation**  
Implements the full ASM pipeline from the proposal:
- Custom `PointGoalEnv` mimicking Safety Gym's interface
- Recovery RL with a **static ε** (Conservative & Aggressive baselines)
- Recovery RL with **ASM** (sigmoid curriculum over ε)
- Ablation: ASM-Sigmoid vs ASM-Linear vs ASM-Step
- Metrics: cumulative cost, episode return, steps-to-first-success, Safety-Return Pareto frontier


In [ ]:
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.distributions import Normal
import collections
import random
import math
import matplotlib
matplotlib.use('Agg')          # headless-safe; remove if running interactively
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from dataclasses import dataclass, field
from typing import List, Optional, Tuple, Dict

torch.manual_seed(42)
np.random.seed(42)
random.seed(42)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {DEVICE}")
print(f"PyTorch {torch.__version__}")


## 1. Environment — Custom PointGoal

A 2D point-navigation task with randomised hazard circles. The agent must reach a goal while avoiding hazards. This mirrors the Safety Gym `PointGoal1` interface so the code can be swapped to `safety_gymnasium.make('SafetyPointGoal1-v0')` when that library is available.

In [ ]:
class PointGoalEnv:
    """
    2D point-goal environment mimicking the Safety Gym interface.

    Observation (8-dim):
        [pos_x, pos_y, goal_dx, goal_dy, nearest_haz_dx, nearest_haz_dy,
         dist_to_goal, dist_to_nearest_haz]

    Action (2-dim, clipped to [-1, 1]):
        [vel_x, vel_y] — scaled by step_size

    Safety signal:
        info['cost'] = 1.0 whenever agent overlaps a hazard circle
    """

    def __init__(
        self,
        n_hazards: int = 3,
        hazard_radius: float = 0.2,
        goal_radius: float = 0.1,
        step_size: float = 0.05,
        max_steps: int = 200,
        seed: int = 0,
    ):
        self.n_hazards = n_hazards
        self.hazard_radius = hazard_radius
        self.goal_radius = goal_radius
        self.step_size = step_size
        self.max_steps = max_steps
        self._rng = np.random.default_rng(seed)

        # Gymnasium-compatible space stubs
        self.observation_space = type("Space", (), {"shape": (8,)})()
        self.action_space = type("Space", (), {"shape": (2,), "low": -1.0, "high": 1.0})()

    # ------------------------------------------------------------------
    def reset(self, seed: Optional[int] = None) -> Tuple[np.ndarray, dict]:
        self._step = 0
        self._pos = np.zeros(2, dtype=np.float32)

        # Randomise goal position (ring between 0.8 and 1.2 from origin)
        angle = self._rng.uniform(0, 2 * math.pi)
        dist = self._rng.uniform(0.8, 1.2)
        self._goal = np.array([dist * math.cos(angle), dist * math.sin(angle)], dtype=np.float32)

        # Randomise hazard positions (ring between 0.15 and 0.85 from origin)
        h_angles = self._rng.uniform(0, 2 * math.pi, self.n_hazards)
        h_dists = self._rng.uniform(0.15, 0.85, self.n_hazards)
        self._hazards = np.stack(
            [h_dists * np.cos(h_angles), h_dists * np.sin(h_angles)], axis=1
        ).astype(np.float32)

        return self._get_obs(), {}

    # ------------------------------------------------------------------
    def step(self, action: np.ndarray) -> Tuple[np.ndarray, float, bool, bool, dict]:
        action = np.clip(action, -1.0, 1.0)
        self._pos += action * self.step_size
        self._step += 1

        dist_goal = float(np.linalg.norm(self._goal - self._pos))
        haz_dists = [float(np.linalg.norm(h - self._pos)) for h in self._hazards]
        in_hazard = any(d < self.hazard_radius for d in haz_dists)

        # Reward: dense proximity + goal bonus
        reward = -dist_goal + (5.0 if dist_goal < self.goal_radius else 0.0)
        cost = 1.0 if in_hazard else 0.0
        done = dist_goal < self.goal_radius
        truncated = self._step >= self.max_steps

        return self._get_obs(), reward, done, truncated, {"cost": cost}

    # ------------------------------------------------------------------
    def _get_obs(self) -> np.ndarray:
        goal_vec = self._goal - self._pos
        haz_vecs = self._hazards - self._pos
        haz_dists = np.linalg.norm(haz_vecs, axis=1)
        nearest = np.argmin(haz_dists)
        nearest_vec = haz_vecs[nearest]
        obs = np.concatenate([
            self._pos,
            goal_vec,
            nearest_vec,
            [np.linalg.norm(goal_vec), haz_dists[nearest]],
        ])
        return obs.astype(np.float32)


# Quick sanity check
_env = PointGoalEnv()
_obs, _ = _env.reset()
_obs2, _r, _d, _t, _info = _env.step(np.array([0.5, 0.5]))
assert _obs.shape == (8,), f"Bad obs shape: {_obs.shape}"
assert "cost" in _info
print(f"PointGoalEnv OK — obs: {_obs.shape}, reward sample: {_r:.3f}, cost: {_info['cost']}")


## 2. Infrastructure

In [ ]:
class ReplayBuffer:
    """Fixed-capacity experience replay for off-policy SAC training."""

    def __init__(self, capacity: int = 100_000):
        self.buffer: collections.deque = collections.deque(maxlen=capacity)

    def push(
        self,
        state: np.ndarray,
        action: np.ndarray,
        reward: float,
        next_state: np.ndarray,
        done: float,
        cost: float,
    ):
        self.buffer.append((state, action, reward, next_state, done, cost))

    def sample(self, batch_size: int):
        batch = random.sample(self.buffer, batch_size)
        s, a, r, ns, d, c = zip(*batch)
        return (
            torch.FloatTensor(np.array(s)).to(DEVICE),
            torch.FloatTensor(np.array(a)).to(DEVICE),
            torch.FloatTensor(np.array(r)).to(DEVICE),
            torch.FloatTensor(np.array(ns)).to(DEVICE),
            torch.FloatTensor(np.array(d)).to(DEVICE),
            torch.FloatTensor(np.array(c)).to(DEVICE),
        )

    def __len__(self) -> int:
        return len(self.buffer)


In [ ]:
# ─── Neural Networks ──────────────────────────────────────────────────────────

class SafetyCritic(nn.Module):
    """
    Predicts Q_risk(s,a) — the discounted probability of future constraint
    violation. Output is in [0, 1] via Sigmoid (binary probability).
    """
    def __init__(self, state_dim: int, action_dim: int, hidden: int = 256):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(state_dim + action_dim, hidden), nn.ReLU(),
            nn.Linear(hidden, hidden),                  nn.ReLU(),
            nn.Linear(hidden, 1),                       nn.Sigmoid(),
        )

    def forward(self, state: torch.Tensor, action: torch.Tensor) -> torch.Tensor:
        return self.net(torch.cat([state, action], dim=-1))


class TaskCritic(nn.Module):
    """Twin Q-networks (SAC) to reduce overestimation bias."""
    def __init__(self, state_dim: int, action_dim: int, hidden: int = 256):
        super().__init__()
        def _mlp(sd, ad, h):
            return nn.Sequential(
                nn.Linear(sd + ad, h), nn.ReLU(),
                nn.Linear(h, h),       nn.ReLU(),
                nn.Linear(h, 1),
            )
        self.q1 = _mlp(state_dim, action_dim, hidden)
        self.q2 = _mlp(state_dim, action_dim, hidden)

    def forward(
        self, state: torch.Tensor, action: torch.Tensor
    ) -> Tuple[torch.Tensor, torch.Tensor]:
        sa = torch.cat([state, action], dim=-1)
        return self.q1(sa), self.q2(sa)


class TaskPolicy(nn.Module):
    """
    Stochastic Gaussian policy with tanh squashing (SAC).

    Returns: (action ∈ [-1,1]^d, log_prob)
    """
    LOG_STD_MIN, LOG_STD_MAX = -20, 2

    def __init__(self, state_dim: int, action_dim: int, hidden: int = 256):
        super().__init__()
        self.fc = nn.Sequential(
            nn.Linear(state_dim, hidden), nn.ReLU(),
            nn.Linear(hidden, hidden),    nn.ReLU(),
        )
        self.mu      = nn.Linear(hidden, action_dim)
        self.log_std = nn.Linear(hidden, action_dim)

    def forward(self, state: torch.Tensor) -> Tuple[torch.Tensor, torch.Tensor]:
        x = self.fc(state)
        mu  = self.mu(x)
        std = torch.exp(torch.clamp(self.log_std(x), self.LOG_STD_MIN, self.LOG_STD_MAX))
        dist = Normal(mu, std)
        z    = dist.rsample()
        a    = torch.tanh(z)
        # Corrected log-prob for tanh squashing
        log_prob = dist.log_prob(z) - torch.log(1 - a.pow(2) + 1e-6)
        return a, log_prob.sum(-1, keepdim=True)


## 3. ASM Core Components (Equations 1–3 from paper)

In [ ]:
class ASMScheduler:
    """
    Equation (1): ε_t = ε_min + (ε_max − ε_min) · σ(κ · (ρ_t − τ))

    Enforces a burn-in period (episodes ≤ T_burn) where ε_t = ε_min.
    """

    def __init__(
        self,
        e_min: float  = 0.05,
        e_max: float  = 0.50,
        kappa: float  = 10.0,
        tau:   float  = 0.60,
        burn_in: int  = 50,
        schedule: str = "sigmoid",   # "sigmoid" | "linear" | "step"
        total_episodes: int = 500,
    ):
        self.e_min, self.e_max = e_min, e_max
        self.kappa, self.tau   = kappa, tau
        self.burn_in           = burn_in
        self.schedule          = schedule
        self.total_episodes    = total_episodes

    def get_threshold(self, episode: int, rho: float) -> float:
        if episode <= self.burn_in:
            return self.e_min                          # strict safety floor

        if self.schedule == "sigmoid":
            # Competence-driven adaptive threshold (proposed ASM)
            sig = 1.0 / (1.0 + math.exp(-self.kappa * (rho - self.tau)))
            return self.e_min + (self.e_max - self.e_min) * sig

        elif self.schedule == "linear":
            # Ablation: linear ramp over episodes (ignores competence)
            progress = (episode - self.burn_in) / max(1, self.total_episodes - self.burn_in)
            return self.e_min + (self.e_max - self.e_min) * min(progress, 1.0)

        elif self.schedule == "step":
            # Ablation: discrete steps at competence milestones
            if   rho < 0.2: return self.e_min
            elif rho < 0.4: return self.e_min + (self.e_max - self.e_min) * 0.25
            elif rho < 0.6: return self.e_min + (self.e_max - self.e_min) * 0.50
            elif rho < 0.8: return self.e_min + (self.e_max - self.e_min) * 0.75
            else:           return self.e_max

        else:
            raise ValueError(f"Unknown schedule: {self.schedule}")


class CompetenceTracker:
    """
    Equations (2)-(3): EWMA of binary success signal.
        s_e = 1 if goal reached without violation, else 0
        ρ_e = α·s_e + (1−α)·ρ_{e−1}
    """

    def __init__(self, alpha: float = 0.05):
        self.alpha = alpha
        self.rho   = 0.0

    def update(self, success: bool) -> float:
        s_e      = 1.0 if success else 0.0
        self.rho = self.alpha * s_e + (1 - self.alpha) * self.rho
        return self.rho

    def reset(self):
        self.rho = 0.0


# ── Threshold schedule visualisation ──────────────────────────────────────────
rho_vals = np.linspace(0, 1, 200)
sched_sig  = ASMScheduler(schedule="sigmoid")
sched_lin  = ASMScheduler(schedule="linear", total_episodes=500)
sched_step = ASMScheduler(schedule="step")

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Left: epsilon vs rho (competence-driven)
eps_sig  = [sched_sig.get_threshold(100, r) for r in rho_vals]
eps_step = [sched_step.get_threshold(100, r) for r in rho_vals]
axes[0].plot(rho_vals, eps_sig,  label="ASM-Sigmoid", linewidth=2)
axes[0].plot(rho_vals, eps_step, label="ASM-Step",    linewidth=2, linestyle="--")
axes[0].axhline(0.05, color="red",   linestyle=":", label="Conservative (ε=0.05)")
axes[0].axhline(0.50, color="green", linestyle=":", label="Aggressive   (ε=0.50)")
axes[0].set_xlabel("Competence ρ"); axes[0].set_ylabel("Safety threshold ε")
axes[0].set_title("Threshold vs Competence (post-burn-in)")
axes[0].legend(); axes[0].grid(True, alpha=0.3)

# Right: epsilon vs episode (linear ablation)
eps_range = np.arange(1, 501)
eps_lin   = [sched_lin.get_threshold(e, 0.0) for e in eps_range]
axes[1].plot(eps_range, eps_lin, label="ASM-Linear", linewidth=2, color="orange")
axes[1].axvline(50, color="grey", linestyle=":", label="Burn-in end")
axes[1].set_xlabel("Episode"); axes[1].set_ylabel("Safety threshold ε")
axes[1].set_title("Linear Schedule vs Episode")
axes[1].legend(); axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig("asm_schedules.png", dpi=120)
plt.show()
print("Schedule visualisation saved → asm_schedules.png")


## 4. ASM Agent — SAC with Safety Critic

In [ ]:
class ASM_Agent:
    """
    Recovery RL agent (Algorithm 1 from paper):
    - Task policy: SAC with twin critics
    - Safety critic: binary cross-entropy on cost signal
    - Recovery action: gradient descent on safety critic to find safer nearby action
    """

    BATCH_SIZE    = 256
    GAMMA         = 0.99
    TAU_SOFT      = 0.005    # soft target update coefficient
    ALPHA_ENTROPY = 0.2      # SAC entropy temperature
    RECOVERY_STEPS = 10      # gradient steps for recovery action optimisation

    def __init__(self, state_dim: int, action_dim: int):
        self.state_dim  = state_dim
        self.action_dim = action_dim

        self.policy         = TaskPolicy(state_dim, action_dim).to(DEVICE)
        self.critic         = TaskCritic(state_dim, action_dim).to(DEVICE)
        self.critic_target  = TaskCritic(state_dim, action_dim).to(DEVICE)
        self.critic_target.load_state_dict(self.critic.state_dict())
        self.safety_critic  = SafetyCritic(state_dim, action_dim).to(DEVICE)

        self.p_opt = optim.Adam(self.policy.parameters(),        lr=3e-4)
        self.c_opt = optim.Adam(self.critic.parameters(),        lr=3e-4)
        self.s_opt = optim.Adam(self.safety_critic.parameters(), lr=3e-4)

    # ------------------------------------------------------------------
    @torch.no_grad()
    def act(self, state: np.ndarray) -> np.ndarray:
        """Greedy (eval-time) action — no recovery logic applied."""
        st = torch.FloatTensor(state).unsqueeze(0).to(DEVICE)
        a, _ = self.policy(st)
        return a.squeeze().cpu().numpy()

    # ------------------------------------------------------------------
    def get_recovery_action(
        self, state_t: torch.Tensor, action_task: torch.Tensor
    ) -> torch.Tensor:
        """
        Gradient descent on safety critic to find a lower-risk action
        in the neighbourhood of the task action.
        """
        rec = action_task.clone().detach().requires_grad_(True)
        opt = optim.SGD([rec], lr=0.1)
        for _ in range(self.RECOVERY_STEPS):
            risk = self.safety_critic(state_t, torch.tanh(rec))
            opt.zero_grad()
            risk.backward()
            opt.step()
        return torch.tanh(rec).detach()

    # ------------------------------------------------------------------
    def update(self, buffer: ReplayBuffer) -> Dict[str, float]:
        """One gradient step on all three networks."""
        if len(buffer) < self.BATCH_SIZE:
            return {}

        s, a, r, ns, d, c = buffer.sample(self.BATCH_SIZE)

        # ── 1. Safety Critic (BCE) ─────────────────────────────────────
        risk_pred = self.safety_critic(s, a)
        s_loss = F.binary_cross_entropy(risk_pred, c.unsqueeze(1).clamp(0.0, 1.0))
        self.s_opt.zero_grad(); s_loss.backward(); self.s_opt.step()

        # ── 2. Task Critic (SAC, Bellman) ──────────────────────────────
        with torch.no_grad():
            na, nlp    = self.policy(ns)
            q1t, q2t   = self.critic_target(ns, na)
            target_q   = r.unsqueeze(1) + (1 - d.unsqueeze(1)) * self.GAMMA * (
                torch.min(q1t, q2t) - self.ALPHA_ENTROPY * nlp
            )
        cq1, cq2 = self.critic(s, a)
        c_loss = F.mse_loss(cq1, target_q) + F.mse_loss(cq2, target_q)
        self.c_opt.zero_grad(); c_loss.backward(); self.c_opt.step()

        # ── 3. Policy (SAC, entropy-regularised) ───────────────────────
        na2, lp2   = self.policy(s)
        q1p, q2p   = self.critic(s, na2)
        p_loss     = (self.ALPHA_ENTROPY * lp2 - torch.min(q1p, q2p)).mean()
        self.p_opt.zero_grad(); p_loss.backward(); self.p_opt.step()

        # ── 4. Soft update target critic ───────────────────────────────
        for p, tp in zip(self.critic.parameters(), self.critic_target.parameters()):
            tp.data.copy_(self.TAU_SOFT * p.data + (1 - self.TAU_SOFT) * tp.data)

        return {
            "safety_loss": s_loss.item(),
            "critic_loss": c_loss.item(),
            "policy_loss": p_loss.item(),
        }


## 5. Training Loop & Experiment Runner

In [ ]:
@dataclass
class RunResult:
    name: str
    ep_rewards:       List[float] = field(default_factory=list)
    ep_costs:         List[float] = field(default_factory=list)
    cumulative_costs: List[float] = field(default_factory=list)
    ep_thresholds:    List[float] = field(default_factory=list)
    competence:       List[float] = field(default_factory=list)
    steps_to_first_success: Optional[int] = None   # total env steps
    total_steps_log:  List[int]   = field(default_factory=list)  # cumulative steps


def train(
    env_seed:       int  = 42,
    episodes:       int  = 500,
    schedule:       str  = "sigmoid",   # "sigmoid"|"linear"|"step"|"fixed"
    fixed_epsilon:  float = 0.05,       # used only when schedule=="fixed"
    burn_in:        int  = 50,
    name:           str  = "ASM",
    verbose:        bool = True,
    log_interval:   int  = 50,
) -> Tuple[ASM_Agent, RunResult]:
    """
    Full training loop implementing Algorithm 1.
    Returns trained agent and logged metrics.
    """
    env     = PointGoalEnv(seed=env_seed)
    sd, ad  = env.observation_space.shape[0], env.action_space.shape[0]
    agent   = ASM_Agent(sd, ad)
    buffer  = ReplayBuffer(100_000)
    tracker = CompetenceTracker(alpha=0.05)
    result  = RunResult(name=name)

    if schedule == "fixed":
        scheduler = None
    else:
        scheduler = ASMScheduler(
            e_min=0.05, e_max=0.50, kappa=10.0, tau=0.60,
            burn_in=burn_in, schedule=schedule, total_episodes=episodes,
        )

    cumulative_cost  = 0.0
    total_steps      = 0
    first_success_ep = None

    for ep in range(1, episodes + 1):
        state, _ = env.reset()
        ep_reward, ep_cost = 0.0, 0.0
        reached_goal, violated = False, False

        # Determine threshold for this episode
        eps = fixed_epsilon if scheduler is None else scheduler.get_threshold(ep, tracker.rho)

        for t in range(env.max_steps):
            st = torch.FloatTensor(state).unsqueeze(0).to(DEVICE)
            with torch.no_grad():
                at_task, _ = agent.policy(st)

            # ── ASM Switching Logic (line 14-16 of Algorithm 1) ─────
            if agent.safety_critic(st, at_task).item() > eps:
                action = agent.get_recovery_action(st, at_task).squeeze().cpu().numpy()
            else:
                action = at_task.squeeze().cpu().numpy()

            ns, r, done, trunc, info = env.step(action)
            cost = float(info.get("cost", 0.0))

            buffer.push(state, action, r, ns, float(done or trunc), cost)
            state       = ns
            ep_reward  += r
            ep_cost    += cost
            total_steps += 1

            if cost > 0:
                violated = True
            if done:
                reached_goal = True

            # Update networks at every step once buffer is warm
            if len(buffer) >= agent.BATCH_SIZE:
                agent.update(buffer)

            if done or trunc:
                break

        # ── Post-episode bookkeeping ─────────────────────────────────
        success = reached_goal and not violated
        tracker.update(success)

        if success and first_success_ep is None:
            first_success_ep = total_steps
            result.steps_to_first_success = total_steps

        cumulative_cost += ep_cost
        result.ep_rewards.append(ep_reward)
        result.ep_costs.append(ep_cost)
        result.cumulative_costs.append(cumulative_cost)
        result.ep_thresholds.append(eps)
        result.competence.append(tracker.rho)
        result.total_steps_log.append(total_steps)

        if verbose and ep % log_interval == 0:
            avg_r = np.mean(result.ep_rewards[-log_interval:])
            avg_c = np.mean(result.ep_costs[-log_interval:])
            print(
                f"[{name}] Ep {ep:4d}/{episodes} | ε={eps:.3f} | "
                f"ρ={tracker.rho:.3f} | Avg Rew={avg_r:+.1f} | "
                f"Avg Cost={avg_c:.2f} | Cum Cost={cumulative_cost:.0f}"
            )

    return agent, result


## 6. Experiments

### 6a. Main comparison: Conservative vs ASM-Sigmoid vs Aggressive

In [ ]:
EPISODES = 500
SEED     = 42

print("=" * 65)
print("Training Conservative baseline (fixed ε = 0.05) ...")
print("=" * 65)
_, res_conservative = train(
    env_seed=SEED, episodes=EPISODES, schedule="fixed",
    fixed_epsilon=0.05, name="Conservative", log_interval=100,
)

print()
print("=" * 65)
print("Training ASM-Sigmoid (proposed method) ...")
print("=" * 65)
agent_asm, res_asm_sigmoid = train(
    env_seed=SEED, episodes=EPISODES, schedule="sigmoid",
    burn_in=50, name="ASM-Sigmoid", log_interval=100,
)

print()
print("=" * 65)
print("Training Aggressive baseline (fixed ε = 0.50) ...")
print("=" * 65)
_, res_aggressive = train(
    env_seed=SEED, episodes=EPISODES, schedule="fixed",
    fixed_epsilon=0.50, name="Aggressive", log_interval=100,
)

print("\nAll main runs complete.")


### 6b. Ablation — Sigmoid vs Linear vs Step

In [ ]:
print("=" * 65)
print("Ablation: ASM-Linear ...")
print("=" * 65)
_, res_asm_linear = train(
    env_seed=SEED, episodes=EPISODES, schedule="linear",
    burn_in=50, name="ASM-Linear", log_interval=100,
)

print()
print("=" * 65)
print("Ablation: ASM-Step ...")
print("=" * 65)
_, res_asm_step = train(
    env_seed=SEED, episodes=EPISODES, schedule="step",
    burn_in=50, name="ASM-Step", log_interval=100,
)

print("\nAblation runs complete.")


## 7. Metrics & Visualisation

In [ ]:
def smooth(x: List[float], window: int = 20) -> np.ndarray:
    """Simple moving-average smoother for noisy RL curves."""
    x = np.array(x, dtype=float)
    if len(x) < window:
        return x
    kernel = np.ones(window) / window
    return np.convolve(x, kernel, mode="same")


def plot_main_comparison(results: List[RunResult], save: str = "main_comparison.png"):
    colours = {"Conservative": "#e74c3c", "ASM-Sigmoid": "#2ecc71", "Aggressive": "#3498db"}
    eps_arr = np.arange(EPISODES)

    fig = plt.figure(figsize=(18, 10))
    gs  = gridspec.GridSpec(2, 3, figure=fig, hspace=0.40, wspace=0.35)

    # 1. Episode Return
    ax1 = fig.add_subplot(gs[0, 0])
    for r in results:
        ax1.plot(smooth(r.ep_rewards), label=r.name, color=colours.get(r.name), lw=2)
    ax1.set_title("Episode Return (smoothed)"); ax1.set_xlabel("Episode"); ax1.set_ylabel("Return")
    ax1.legend(); ax1.grid(True, alpha=0.3)

    # 2. Cumulative Cost
    ax2 = fig.add_subplot(gs[0, 1])
    for r in results:
        ax2.plot(r.cumulative_costs, label=r.name, color=colours.get(r.name), lw=2)
    ax2.set_title("Cumulative Safety Cost"); ax2.set_xlabel("Episode"); ax2.set_ylabel("Cumulative Cost")
    ax2.legend(); ax2.grid(True, alpha=0.3)

    # 3. Adaptive Threshold (ASM only)
    ax3 = fig.add_subplot(gs[0, 2])
    asm_r = next(r for r in results if r.name == "ASM-Sigmoid")
    ax3.plot(asm_r.ep_thresholds, color="#2ecc71", lw=2, label="ε_t")
    ax3.plot(asm_r.competence,    color="#f39c12", lw=2, linestyle="--", label="ρ_t (competence)")
    ax3.axvline(50, color="grey", linestyle=":", label="Burn-in end")
    ax3.set_title("ASM: Threshold & Competence"); ax3.set_xlabel("Episode")
    ax3.legend(); ax3.grid(True, alpha=0.3)

    # 4. Per-episode cost
    ax4 = fig.add_subplot(gs[1, 0])
    for r in results:
        ax4.plot(smooth(r.ep_costs, 30), label=r.name, color=colours.get(r.name), lw=2)
    ax4.set_title("Per-Episode Cost (smoothed)"); ax4.set_xlabel("Episode"); ax4.set_ylabel("Cost")
    ax4.legend(); ax4.grid(True, alpha=0.3)

    # 5. Safety–Return Pareto frontier
    ax5 = fig.add_subplot(gs[1, 1])
    window = 50
    for r in results:
        rets  = [np.mean(r.ep_rewards[max(0,i-window):i+1]) for i in range(len(r.ep_rewards))]
        costs = [np.mean(r.ep_costs[max(0,i-window):i+1])   for i in range(len(r.ep_costs))]
        ax5.scatter(costs, rets, label=r.name, color=colours.get(r.name), s=8, alpha=0.5)
    ax5.set_title("Safety–Return Pareto Frontier\n(rolling 50-ep window)"); 
    ax5.set_xlabel("Avg Cost (lower = safer)"); ax5.set_ylabel("Avg Return")
    ax5.legend(); ax5.grid(True, alpha=0.3)

    # 6. Steps to first success bar chart
    ax6 = fig.add_subplot(gs[1, 2])
    names = [r.name for r in results]
    steps = [r.steps_to_first_success if r.steps_to_first_success else EPISODES * 200 for r in results]
    bars = ax6.bar(names, steps, color=[colours.get(n,"#95a5a6") for n in names])
    ax6.set_title("Steps to First Successful Episode\n(lower = better exploration)"); 
    ax6.set_ylabel("Total Env Steps")
    ax6.bar_label(bars, fmt="%d", padding=3)
    ax6.grid(True, alpha=0.3, axis="y")

    plt.suptitle("Recovery RL: Adaptive Safety Margins (ASM) — Main Comparison", fontsize=14, fontweight="bold")
    plt.savefig(save, dpi=150, bbox_inches="tight")
    plt.show()
    print(f"Saved → {save}")


plot_main_comparison([res_conservative, res_asm_sigmoid, res_aggressive])


In [ ]:
def plot_ablation(results: List[RunResult], save: str = "ablation.png"):
    colours = {"ASM-Sigmoid": "#2ecc71", "ASM-Linear": "#f39c12", "ASM-Step": "#9b59b6"}

    fig, axes = plt.subplots(1, 3, figsize=(16, 4))

    # Return
    for r in results:
        axes[0].plot(smooth(r.ep_rewards), label=r.name, color=colours.get(r.name), lw=2)
    axes[0].set_title("Episode Return (smoothed)"); axes[0].set_xlabel("Episode")
    axes[0].set_ylabel("Return"); axes[0].legend(); axes[0].grid(True, alpha=0.3)

    # Cumulative cost
    for r in results:
        axes[1].plot(r.cumulative_costs, label=r.name, color=colours.get(r.name), lw=2)
    axes[1].set_title("Cumulative Cost"); axes[1].set_xlabel("Episode")
    axes[1].legend(); axes[1].grid(True, alpha=0.3)

    # Threshold schedule over episodes
    for r in results:
        axes[2].plot(r.ep_thresholds, label=r.name, color=colours.get(r.name), lw=2)
    axes[2].axvline(50, color="grey", linestyle=":", label="Burn-in end")
    axes[2].set_title("Threshold ε over Episodes"); axes[2].set_xlabel("Episode")
    axes[2].set_ylabel("ε"); axes[2].legend(); axes[2].grid(True, alpha=0.3)

    plt.suptitle("Ablation Study: Schedule Shape", fontsize=13, fontweight="bold")
    plt.tight_layout()
    plt.savefig(save, dpi=150, bbox_inches="tight")
    plt.show()
    print(f"Saved → {save}")


plot_ablation([res_asm_sigmoid, res_asm_linear, res_asm_step])


## 8. Summary Statistics

In [ ]:
def summary_table(results: List[RunResult]):
    # Last-100-episode window
    W = 100
    header = f"{'Method':<18} | {'Avg Return':>11} | {'Avg Cost':>9} | {'Cum Cost':>10} | {'Steps-1st-Success':>18}"
    print(header)
    print("-" * len(header))
    for r in results:
        avg_rew  = np.mean(r.ep_rewards[-W:])
        avg_cost = np.mean(r.ep_costs[-W:])
        cum_cost = r.cumulative_costs[-1]
        s1s      = r.steps_to_first_success or -1
        print(f"{r.name:<18} | {avg_rew:>+11.2f} | {avg_cost:>9.3f} | {cum_cost:>10.0f} | {s1s:>18}")


print("\n=== Main Comparison (last 100 episodes) ===")
summary_table([res_conservative, res_asm_sigmoid, res_aggressive])

print("\n=== Ablation Study (last 100 episodes) ===")
summary_table([res_asm_sigmoid, res_asm_linear, res_asm_step])


## 9. Deterministic Evaluation of Trained ASM Agent

In [ ]:
def evaluate(agent: ASM_Agent, n_episodes: int = 20, env_seed: int = 99) -> dict:
    """
    Greedy (no recovery) evaluation of the task policy.
    Uses a different seed than training to test generalisation.
    """
    env = PointGoalEnv(seed=env_seed)
    rewards, costs, successes = [], [], []
    for _ in range(n_episodes):
        state, _ = env.reset()
        ep_r, ep_c, goal, viol = 0.0, 0.0, False, False
        for _ in range(env.max_steps):
            action = agent.act(state)
            state, r, done, trunc, info = env.step(action)
            ep_r += r; ep_c += info.get("cost", 0.0)
            if info.get("cost", 0.0) > 0: viol = True
            if done: goal = True
            if done or trunc: break
        rewards.append(ep_r); costs.append(ep_c)
        successes.append(goal and not viol)

    return {
        "mean_return":  np.mean(rewards),
        "std_return":   np.std(rewards),
        "mean_cost":    np.mean(costs),
        "success_rate": np.mean(successes),
    }


eval_results = evaluate(agent_asm, n_episodes=20)
print("\n=== Trained ASM-Sigmoid Agent — Deterministic Evaluation ===")
for k, v in eval_results.items():
    print(f"  {k:<18}: {v:.4f}")


## 10. Running with Real Safety Gym

When `safety-gymnasium` is available (e.g. on a machine with pygame/SDL2), replace the environment instantiation:

```python
import safety_gymnasium

env = safety_gymnasium.make(
    "SafetyPointGoal1-v0",
    render_mode=None,   # or "rgb_array" for recording
)
```

The rest of the code is **drop-in compatible** — both environments expose identical `reset()`, `step()`, and `observation_space`/`action_space` attributes.
